# 1. Package Imports

In [0]:
import re
import logging
import pyspark.sql.functions as F
from pyspark.sql import Window
from functools import reduce

# 2. Dataset Config

In [0]:

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("Silver_Layer_Dam_Levels")

ds_config ={
    "bronze_table": "cpt_utility_catalog.bronze.bronze_dam_levels_raw",
    "silver_table": "cpt_utility_catalog.silver.silver_dam_levels_cleaned",
    "changes":{
        "headers":{
            "drop_column":{
                "drop": True,
                "columns": ["ObjectId",
                            "TOTAL_STORED___BIG_5_STORAGE",
                            "TOTAL_STORED___BIG_5_Current",
                            "TOTAL_STORED___BIG_5_Last_Year",
                            ]
            }, 
            "rename_headers": True,
        },
        "columns":{
            "trim_whitespace": True,
            "cast_data_types": True, 
            "drop_duplicates": True,
            "new_totals": True,
            "data_types":{
                "date":{
                    "format": "dd-MMM-yy",
                    "type": "date",
                },
                "height_m": "decimal(5,2)",
                "storage_Ml": "decimal(10, 3)",
                "current_pct": "decimal(12,8)",
                "last_year_pct": "decimal(12,8)"
            }
            
        },
    }
}


df_new = spark.table(ds_config["bronze_table"])
hdr_config = ds_config["changes"]["headers"]
col_config = ds_config["changes"]["columns"]

logger.info("Silver layer Dam levels table configuration loaded")


# 3. Dataset Cleaning

## 3.1 Drop Columns

In [0]:
# conditional to drop columns
logger.info("Dropping column(s)")
if hdr_config["drop_column"]["drop"]:
    drop_list = hdr_config["drop_column"]["columns"]
    # dropping columns at once
    if isinstance(drop_list, list):
        df_new = df_new.drop(*drop_list)
logger.info("\t - Column(s) dropped")

## 3.2 Clean Header Names

In [0]:
# conditional to rename headers
if hdr_config["rename_headers"]:
    logger.info("Renaming header(s)")
    
    # renaming headers in [name]_[metric]_[unit] unit format
    renamed_headers = [col.rstrip("_").lower().replace("last","_last")
                    .replace("ë","e").replace("ml","Ml").replace("current", "current_pct")
                    .replace("year","year_pct").replace("___","_")
                    .replace("__","_").replace("steenbras_storage","steenbras_lower_storage") 
                    .replace("6_storage", "6_storage_Ml").replace("woodhead_height", "woodhead_height_m")
                    .replace("victoria_height", "victoria_height_m")
                     for col in df_new.columns]
    print(renamed_headers)
    # creating new dict() old_header_name : new_header_name
    new_header_map = {old_header : new_header for old_header, new_header in zip(df_new.columns, renamed_headers)}

    # renaming all headers at once
    df_new = df_new.withColumnsRenamed(new_header_map)
    logger.info("\t - Header(s) renamed")

## 3.3 Clean Malformed Columns

In [0]:
logger.info("cleaning malformed column cells")
# some columns have double dots, replacing with single dot
df_new = df_new.select([F.regexp_replace(F.col(col), r"\.{2,}", ".").alias(col) for col in df_new.columns])
logger.info("\t - Malformed column cells cleaned")

## 3.4 Trim Whitespace

In [0]:
# conditional to trim columns
if col_config["trim_whitespace"]:
    logger.info("Trimming whitespace")
    # iterating through all columns and trimming whitespace
    df_new = df_new.select([F.regexp_replace(F.col(col), r"\s+", "").alias(col) for col in df_new.columns])
    logger.info("\t - Whitespace trimmed")

## 3.5 Cast Data Types

In [0]:
if col_config["cast_data_types"]:
    logger.info("Casting data types")

    # iterating formatting and casting Datetype to date column
    to_mm_format = df_new.withColumn("date_cleaned", F.regexp_replace(F.lower(F.col("date")), "sept", "sep"))
    to_mm_format = to_mm_format.withColumn("date", F.to_date(F.initcap(F.col("date_cleaned")), col_config["data_types"]["date"]["format"]))
    df_new = to_mm_format.drop("date_cleaned")

    dtype_transformations = dict()
    # criating dictionary of data casting instructions to cast at once below
    for col in df_new.columns:
        if "height_m" in col:
            dtype_transformations[col] = F.col(col).try_cast(col_config["data_types"]["height_m"]).alias(col)
        elif "storage_Ml" in col:
            dtype_transformations[col] = F.col(col).try_cast(col_config["data_types"]["storage_Ml"]).alias(col)
        elif "current_pct" in col:
            dtype_transformations[col] = F.col(col).try_cast(col_config["data_types"]["current_pct"]).alias(col)
        elif "last_year_pct" in col:
            dtype_transformations[col] = F.col(col).try_cast(col_config["data_types"]["last_year_pct"]).alias(col)
       
    df_new = df_new.withColumns(dtype_transformations)
    logger.info("\t - Data types casted")


## 3.6 Recalculating New Totals

In [0]:
if col_config["new_totals"]:
    # since some row cells were found to have malformed data, a recalculation of totals must be done
    logger.info("Calculating new totals")

    #sum of storage for big 6 at row level 
    big_6 = [
    "berg_river_",
    "theewaterskloof_",
    "voelvlei_",
    "wemmershoek_",
    "steenbras_upper_",
    "steenbras_lower_",
]
    # sum of storage for big 6 at row level
    storage_sum = reduce(
        lambda x, y: x + y,
        [F.coalesce(F.col(col + "storage_Ml"), F.lit(0)) for col in big_6]
    )

    #sum of current pct for big 6 at row level 
    current_pct_sum = reduce(
        lambda x, y: x + y,
        [F.coalesce(F.col(col + "current_pct"), F.lit(0)) for col in big_6]
    )


    #sum of last year current pct for big 6 at row level 
    last_year_pct_sum = reduce(
        lambda x, y: x + y,
        [F.coalesce(F.col(col + "last_year_pct"), F.lit(0)) for col in big_6]
    )

    df_new = df_new.withColumn("total_stored_big_6_storage_Ml", storage_sum)
    df_new = df_new.withColumn("total_stored_big_6_current_pct", F.coalesce(current_pct_sum, F.lit(0)) /F.lit(6))
    df_new = df_new.withColumn("total_stored_big_6_last_year_pct", F.coalesce(last_year_pct_sum, F.lit(0))/F.lit(6))
logger.info("\t - Totals calculated")

## 3.7 Dropping Duplicates

In [0]:

logger.info("Dropping Duplicate(s)")

# 1. Check all columns except 'date' for nulls
cols_to_check = [c for c in df_new.columns if c != "date"]

# 2. Count how many NULLs exist in each row across all columns
null_count_expr = sum(
    [F.when(F.col(c).isNull(), 1).otherwise(0) for c in cols_to_check]
)

df_with_nulls = df_new.withColumn("null_count", null_count_expr)

# 3. Create a window grouped by date, ordered by null_count ASCENDING
# (Lowest null count = rank 1)
window_spec = Window.partitionBy("date").orderBy(F.col("null_count").asc())

# 4. Filter to keep only the best row per date and clean up temporary columns
df_deduped = (
    df_with_nulls.withColumn("row_num", F.row_number().over(window_spec))
    .filter(F.col("row_num") == 1)
    .drop("row_num", "null_count")
)

logger.info("\t - Duplicate(s) Dropped")


# 4. Writing To Silver Layer

In [0]:
logger.info("Writing to Silver Table")
(df_new.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(ds_config["silver_table"]))
logger.info("\t - Data written to Silver Table")